# 处理图表、图形与幻灯片
Claude 在处理图表、图形以及更广泛的幻灯片方面能力很强。根据你的使用场景，有一些技巧和窍门值得利用。本篇示例将展示把 Claude 用于这类材料的常见模式。

## 图表与图形
在大多数情况下，把 Claude 用于图表和图形很简单。我们逐步来看如何摄取这些材料并传给 Claude，以及一些提升效果的常用技巧。

### 摄取与调用 Claude API
把图表和图形传给 Claude 的最佳方式，是利用它的视觉能力和 PDF 支持功能。也就是说，给 Claude 一份该图表或图形的 PDF 文档，并附上一个关于它的文本问题。

目前只有 `claude-sonnet-4-6` 支持 PDF 功能。由于该功能仍处于测试阶段，你需要为请求提供 `pdfs-2024-09-25` 测试版请求头。

In [ ]:
# Install and create the Anthropic client.
%pip install anthropic

In [2]:
import base64

from anthropic import Anthropic

# While PDF support is in beta, you must pass in the correct beta header
client = Anthropic(default_headers={"anthropic-beta": "pdfs-2024-09-25"})
# For now, only claude-sonnet-4-6 supports PDFs
MODEL_NAME = "claude-sonnet-4-6"

In [37]:
# Make a useful helper function.
def get_completion(messages):
    response = client.messages.create(
        model=MODEL_NAME, max_tokens=8192, temperature=0, messages=messages
    )
    return response.content[0].text

In [12]:
# To start, we'll need a PDF. We will be using the .pdf document located at cvna_2021_annual_report.pdf.
# Start by reading in the PDF and encoding it as base64.
with open("./documents/cvna_2021_annual_report.pdf", "rb") as pdf_file:
    binary_data = pdf_file.read()
    base_64_encoded_data = base64.b64encode(binary_data)
    base64_string = base_64_encoded_data.decode("utf-8")

我们来看看如何把这份文档连同简单问题一起传给模型。

In [13]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64_string,
                },
            },
            {"type": "text", "text": "What's in this document? Answer in a single sentence."},
        ],
    }
]

print(get_completion(messages))

This is a page from Carvana's 2021 Annual Report showing four key metrics: retail units sold, total revenue, total markets at year end, and car vending machines, all displaying significant growth from 2014 to 2021.


相当不错！现在我们再问一些更有用的问题。

In [15]:
questions = [
    "What was CVNA revenue in 2020?",
    "How many additional markets has Carvana added since 2014?",
    "What was 2016 revenue per retail unit sold?",
]

for index, question in enumerate(questions):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": base64_string,
                    },
                },
                {"type": "text", "text": question},
            ],
        }
    ]

    print(f"\n----------Question {index + 1}----------")
    print(get_completion(messages))


----------Question 1----------
According to the graph showing Total Revenue ($M), Carvana's revenue in 2020 was $5,587 million (or approximately $5.59 billion).

----------Question 2----------
According to the "TOTAL MARKETS AT YEAR END" graph, Carvana started with 4 markets in 2014 and grew to 311 markets by 2021. Therefore, Carvana added 307 markets since 2014 (311 - 4 = 307 additional markets).

----------Question 3----------
Let me calculate this for you:

2016 Revenue: $365 million
2016 Retail Units Sold: 18,761 units

$365 million ÷ 18,761 units = $19,455 per unit (rounded to nearest dollar)

So in 2016, Carvana's revenue per retail unit sold was approximately $19,455.


如你所见，Claude 能够回答关于图表和图形的相当细致的问题。不过，有一些技巧和窍门能帮你把它用到极致。
- 有时 Claude 的算术能力会成为干扰。你会发现，对上面第三个问题多次采样，它偶尔会给出错误的最终答案，因为它把算术算错了。考虑为 Claude 提供一个计算器工具，以避免这类错误。
- 面对极其复杂的图表和图形，我们可以要求 Claude“先描述你在文档中看到的每一个数据点”，这能带来与传统思维链类似的提升效果。
- Claude 有时难以处理依赖大量颜色来传达信息的图表，例如有许多分组的分组柱状图。要求 Claude 先用 HEX 颜色码识别图形中的颜色，可以提高它的准确性。

## 幻灯片
既然我们已经知道 Claude 是图表和图形方面的高手，那么把它扩展到图表和图形的真正主场——幻灯片——就是顺理成章的事。

对许多领域（包括金融服务）而言，幻灯片是一个关键的信息来源。你*可以*使用 PyPDF 之类的包从幻灯片中提取文本，但由于幻灯片以图表为主，这往往不是好选择，因为模型会难以获取它们真正需要的信息。

因此，PDF 支持功能可以成为一个很好的替代方案。它在处理 PDF 文档时同时使用提取的文本和视觉能力。本节将讲解如何在 Claude 中使用 PDF 文档来审阅幻灯片，以及如何处理这一方式的一些常见陷阱。

把典型幻灯片导入 Claude 的最佳方式，是把它下载为 PDF 并直接提供给 Claude。

In [17]:
# Open the multi-page PDF document the same way we did earlier.
with open("./documents/twilio_q4_2023.pdf", "rb") as pdf_file:
    binary_data = pdf_file.read()
    base_64_encoded_data = base64.b64encode(binary_data)
    base64_string = base_64_encoded_data.decode("utf-8")

In [18]:
# Now let's pass the document directly to Claude. Note that Claude will process both the text and visual elements of the document.
question = "What was Twilio y/y revenue growth for fiscal year 2023?"
content = [
    {
        "type": "document",
        "source": {"type": "base64", "media_type": "application/pdf", "data": base64_string},
    },
    {"type": "text", "text": question},
]

messages = [{"role": "user", "content": content}]

print(get_completion(messages))

According to the financial results shown in the presentation, Twilio's year-over-year revenue growth for fiscal year 2023 was 9%. This can be found in the "Total Company Results Highlights" section, which shows FY 2023 revenue growth of 9%.


这种方式是入门的好办法，对某些使用场景而言也是性能最好的选择。但它存在一些限制。
- 一个请求中所有提供的文档合计最多只能包含 100 页（我们计划逐步提高这一限制）。
- 如果你把幻灯片内容用于 RAG，把多模态 PDF 引入嵌入向量可能引发问题

幸运的是，我们可以利用 Claude 的视觉能力，得到幻灯片的、**文本形式**的、质量远高于普通 PDF 文本提取的表示。

我们发现最好的做法是让 Claude 从头到尾按顺序讲解这份幻灯片，把当前幻灯片和它此前的讲解内容一起传给它。下面来看怎么做。

In [41]:
# Define a prompt for narrating our slide deck. We would adjut this prompt based on the nature of the deck, but keep the structure largely the same.
prompt = """
You are the Twilio CFO, narrating your Q4 2023 earnings presentation.

The entire earnings presentation document is provided to you.
Please narrate this presentation from Twilio's Q4 2023 Earnings as if you were the presenter. Do not talk about any things, especially acronyms, if you are not exactly sure you know what they mean.

Do not leave any details un-narrated as some of your viewers are vision-impaired, so if you don't narrate every number they won't know the number.

Structure your response like this:
<narration>
    <page_narration id=1>
    [Your narration for page 1]
    </page_narration>

    <page_narration id=2>
    [Your narration for page 2]
    </page_narration>

    ... and so on for each page
</narration>

Use excruciating detail for each page, ensuring you describe every visual element and number present. Show the full response in a single message.
"""
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64_string,
                },
            },
            {"type": "text", "text": prompt},
        ],
    }
]

# Now we use our prompt to narrate the entire deck. Note that this may take a few minutes to run (often up to 10).
completion = get_completion(messages)

In [42]:
import re

# Next we'll parse the response from Claude using regex
pattern = r"<narration>(.*?)</narration>"
match = re.search(pattern, completion.strip(), re.DOTALL)
if match:
    narration = match.group(1)
else:
    raise ValueError("No narration available. Likely due to the model response being truncated.")

现在我们有了基于文本的讲解（它远非完美，但已经相当不错），我们就能够把这份幻灯片用于任何纯文本工作流。包括向量检索！

作为最后的合理性检查，我们就这套仅使用讲解的方案问几个问题！

In [43]:
questions = [
    "What percentage of q4 total revenue was the Segment business line?",
    "Has the rate of growth of quarterly revenue been increasing or decreasing? Give just an answer.",
    "What was acquisition revenue for the year ended december 31, 2023 (including negative revenues)?",
]

for index, question in enumerate(questions):
    prompt = f"""You are an expert financial analyst analyzing a transcript of Twilio's earnings call.
Here is the transcript:
<transcript>
{narration}
</transcript>

Please answer the following question:
<question>
{question}
</question>"""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    print(f"\n----------Question {index + 1}----------")
    print(get_completion(messages))


----------Question 1----------
Let me calculate this:

Segment revenue in Q4 2023: $75 million
Total revenue in Q4 2023: $1,076 million

$75M ÷ $1,076M = 0.0697 or approximately 7%

Therefore, the Segment business line represented approximately 7% of Twilio's total Q4 2023 revenue.

----------Question 2----------
Decreasing. The transcript shows Q4 2023 revenue growth was 5% year-over-year, while for the full year 2023 revenue growth was 9% year-over-year, indicating a slowing growth rate. Additionally, the Q1 2024 guidance projects even lower growth of 2-3% year-over-year, confirming the declining trend.

----------Question 3----------
Let me help calculate the acquisition revenue for 2023.

From the transcript, we can see:
- Total revenue for 2023: $4,154 million
- Organic revenue for 2023: $4,146 million

Therefore, acquisition revenue would be:
Total Revenue - Organic Revenue = $4,154M - $4,146M = $8 million

So the acquisition revenue for the year ended December 31, 2023 was $8 m

看起来不错！掌握了这些技巧，你就已经准备好把模型应用于幻灯片这类图表密集的内容了。